In this test, I used Claude Haiku 4.5

# zero-shot prompting

In [ ]:
User: Review this code for bugs and security issues.

GitHub Copilot: I've identified several critical bugs and security issues in this Flask code:

## **Critical Security Issues**

### 1. **SQL Injection Vulnerability** ⚠️ CRITICAL
```python
query = "SELECT * FROM users WHERE user_id = " + user_id
```
The query is built by concatenating user input directly. An attacker could pass malicious SQL like `'; DROP TABLE users; --` to execute arbitrary commands.

**Fix**: Use parameterized queries:
```python
query = "SELECT * FROM users WHERE user_id = ?"
cursor.execute(query, (user_id,))
```

## **Bugs**

### 2. **Missing Input Validation**
`request.args.get('id')` returns `None` if 'id' is not provided, causing a TypeError when concatenating strings. Should validate before use:
```python
user_id = request.args.get('id')
if not user_id:
    return {"error": "Missing id parameter"}, 400
```

### 3. **No Error Handling**
- Database query could fail (connection lost, table doesn't exist)
- `cursor.fetchone()` might return `None` if no user is found
- `process_data()` and `format_response()` could raise exceptions

Should wrap in try-except and handle edge cases:
```python
try:
    user_data = cursor.fetchone()
    if not user_data:
        return {"error": "User not found"}, 404
except sqlite3.Error as e:
    return {"error": "Database error"}, 500
```

### 4. **Missing Database Cleanup**
The connection is never closed. Add a teardown handler:
```python
@app.teardown_appcontext
def close_db(error):
    db = g.pop('db', None)
    if db is not None:
        db.close()
```

## **Code Quality**

### 5. **Poor Variable Names**
`temp` and `res` are cryptic. Use descriptive names:
- `temp` → `processed_data`
- `res` → `response` or `formatted_response`

### 6. **Missing Database Schema**
No guarantee the `users` table exists. Consider adding initialization or at least documenting the expected schema.

# few-shot prompting

In [ ]:
User:  I want you to act as a Senior Python Developer. Here are examples of the feedback style I expect:
Example 1: 'On line 12, instead of manually opening a connection, consider using a context manager (with) to ensure the DB connection closes automatically.'
Example 2: 'On line 19, the string concatenation is a SQL injection vulnerability. Use parameterized queries: cursor.execute("... WHERE id = ?", (user_id,)).'
Example 3: 'On line 24, the variable name temp is vague. Rename it to user_profile to match the data it holds.'Now, audit this code: import os
from flask import Flask, request, g
import sqlite3


app = Flask(__name__)
app.config['DATABASE'] = 'users.db'


def get_db():
    if 'db' not in g:
        g.db = sqlite3.connect(app.config['DATABASE'])
    return g.db


@app.route('/user')
def get_user_profile():
    user_id = request.args.get('id')
    db = get_db()
    cursor = db.cursor()
    
    # Oh boy...
    query = "SELECT * FROM users WHERE user_id = " + user_id
    cursor.execute(query) # This is the problem
    
    user_data = cursor.fetchone()
    
    # Also, what are 'temp' and 'res'?
    temp = process_data(user_data) 
    res = format_response(temp)
    
    return res

GitHub Copilot: # Code Audit - Flask SQLite Application

Here's my senior-level feedback on this code:

---

**Line 10–13** (`get_db()` function): While this leverages Flask's `g` object for connection management, consider adding a teardown handler to explicitly close connections. Currently, closure relies on Flask's automatic cleanup. For clarity and robustness:
```python
@app.teardown_appcontext
def close_connection(exception):
    db = getattr(g, '_database', None)
    if db is not None:
        db.close()
```

**Line 21** (CRITICAL): This is a **SQL injection vulnerability**. Direct string concatenation with user input is extremely dangerous. Always use parameterized queries to safely escape inputs:
```python
cursor.execute("SELECT * FROM users WHERE user_id = ?", (user_id,))
```

**Line 18**: Add input validation before querying. `request.args.get('id')` returns a string, which should be validated:
```python
user_id = request.args.get('id')
if not user_id:
    return {'error': 'Missing user_id'}, 400
```

**Line 25** (`temp`): This variable name is vague and non-descriptive. Rename to clarify intent—something like `processed_user_data` or `user_profile` would make the code self-documenting.

**Line 26** (`res`): Same issue—`res` is ambiguous. Rename to `response` or `formatted_response`.

**Line 24**: Add defensive programming. `cursor.fetchone()` returns `None` if no user is found. Process `user_data` safely:
```python
if user_data is None:
    return {'error': 'User not found'}, 404
```

**General**: Add error handling around database operations (connection failures, malformed queries, etc.) and consider adding type hints and docstrings for maintainability.